# Тепловые карты: MAE (регрессия) и ROC-AUC при ee ≥ 90% (классификация)

Два рисунка, панели расположены горизонтально в порядке **Random → Catalyst (GroupKFold-10) → LORGO**:

1. `heatmap_regression_MAE` — только MAE (столбцы R² убраны);
2. `heatmap_classification_AUC_ee90` — только порог ee ≥ 90%.

В ячейке — точечная оценка ± половина ширины 95% доверительного интервала (кластерный бутстреп из
`metrics_summary.csv`), полуширина — второй строкой в ячейке. Рисунок строится в печатном размере
(ширина 180 мм), размеры шрифтов заданы в пунктах в первой ячейке. Dummy не показан. Цвет MAE: одна шкала на три панели (по умолчанию `Blues_r`: темнее — меньше ошибка, то есть
лучше, как у ROC-AUC). Цвет ROC-AUC: `RdBu` с центром 0,5, общая шкала для трёх панелей.

Ноутбук лежит в папке `regression/`; путь к классификации задан относительно неё.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm

# ----------------------------- CONFIGURATION -----------------------------
REG_CSV = Path("../results/regression/metrics_summary.csv")   # relative to notebooks/
CLF_CSV = Path("../results/classification/metrics_summary.csv")
OUT_DIR = Path("../results/figures/figures_heatmaps_MAE_AUC90")

CLF_THRESHOLD = 90
DPI_PNG = 600

# Рисунок строится сразу в печатном размере, поэтому шрифты заданы в пунктах и не уменьшаются при вставке.
FIG_WIDTH_MM  = 180      # ширина на всю полосу
FIG_HEIGHT_MM = 125      # высота; больше — выше ячейки
FS_VALUE = 7             # значение в ячейке
FS_CI    = 5.5           # «± полуширина ДИ» под значением
FS_TICK  = 7             # подписи осей
FS_TITLE = 7.5           # заголовки панелей

# MAE: одна шкала на все три панели. "Blues_r" — тёмный = меньше ошибка (лучше), как у ROC-AUC;
# "Blues" — тёмный = больше ошибка.
MAE_CMAP = "Blues_r"
# -------------------------------------------------------------------------

SPLITS = ["random", "gkf10_catalyst", "lorgo_group"]
SPLIT_LABELS = {"random": "Random (5-fold × 5)",
                "gkf10_catalyst": "GroupKFold-10 (catalyst)",
                "lorgo_group": "LORGO (research group)"}
REP_ORDER = ["physchem", "maccs", "ecfp4", "ecfp4_count", "drfp", "onehot", "dft", "cgr"]
REP_LABELS = {"physchem": "Physchem", "maccs": "MACCS", "ecfp4": "ECFP4", "ecfp4_count": "ECFP4-count",
              "drfp": "DRFP", "onehot": "OHE", "dft": "DFT", "cgr": "CGR + catalyst"}
REG_MODELS = ["ridge", "svr", "gp", "rf", "xgb", "chemprop"]
CLF_MODELS = ["logreg", "svc", "rf", "xgb", "chemprop"]
MODEL_LABELS = {"ridge": "Ridge", "svr": "SVR", "gp": "GP", "rf": "RF", "xgb": "XGBoost",
                "chemprop": "D-MPNN", "logreg": "LogReg", "svc": "SVC"}

MM = 1 / 25.4
MISSING = "#f0f0f0"
TEXT_DARK = "#222222"
OUT_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Inter", "Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42, "svg.fonttype": "none",
})

reg = pd.read_csv(REG_CSV)
clf = pd.read_csv(CLF_CSV)
reg = reg[reg.model != "dummy"]
clf = clf[(clf.model != "dummy") & (clf.threshold == CLF_THRESHOLD)]
print("регрессия:", reg.shape, "| классификация, порог", CLF_THRESHOLD, ":", clf.shape)

## Вспомогательные функции

In [ ]:
def grid(df, col, models):
    """Матрица «представление × модель» для одного столбца."""
    return (df.pivot(index="representation", columns="model", values=col)
              .reindex(index=REP_ORDER, columns=models).to_numpy(float))


def heat(ax, df, metric, models, cmap, norm, title, ylabels=True):
    M = grid(df, metric, models)
    half = (grid(df, f"{metric}_ci_high", models) - grid(df, f"{metric}_ci_low", models)) / 2
    im = ax.imshow(np.ma.masked_invalid(M), cmap=cmap, norm=norm, aspect="auto")
    ax.set_facecolor(MISSING)
    for (i, j), v in np.ndenumerate(M):
        if np.isnan(v):
            continue
        r, g, b, _ = im.cmap(im.norm(v))
        col = "white" if 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.5 else TEXT_DARK
        ax.text(j, i - 0.14, f"{v:.2f}", ha="center", va="center", fontsize=FS_VALUE, color=col)
        ax.text(j, i + 0.22, f"± {half[i, j]:.2f}", ha="center", va="center", fontsize=FS_CI, color=col)
    ax.set_xticks(range(len(models)), [MODEL_LABELS[m] for m in models], rotation=40, ha="right",
                  rotation_mode="anchor")
    ax.set_yticks(range(len(REP_ORDER)), [REP_LABELS[r] for r in REP_ORDER] if ylabels else [])
    ax.tick_params(length=0, labelsize=FS_TICK, pad=2)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(title, fontsize=FS_TITLE, pad=4)
    return im


def colorbar(fig, im, ax, label=None, shrink=1.0, aspect=30):
    cb = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.045, pad=0.02, aspect=aspect, shrink=shrink)
    cb.ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(5))
    cb.ax.tick_params(labelsize=FS_TICK - 1, length=2, pad=1)
    cb.outline.set_visible(False)
    if label:
        cb.set_label(label, fontsize=FS_TICK)
    return cb


def save(fig, stem):
    for ext in ("pdf", "svg"):
        fig.savefig(OUT_DIR / f"{stem}.{ext}", bbox_inches="tight")
    fig.savefig(OUT_DIR / f"{stem}.png", dpi=DPI_PNG, bbox_inches="tight")

## 1. Регрессия: MAE

In [ ]:
norm = Normalize(reg["MAE"].min(), reg["MAE"].max())   # общая шкала для трёх схем

fig, axes = plt.subplots(1, 3, figsize=(FIG_WIDTH_MM * MM, FIG_HEIGHT_MM * MM), constrained_layout=True)
for k, (ax, s) in enumerate(zip(axes, SPLITS)):
    im = heat(ax, reg[reg.split == s], "MAE", REG_MODELS, MAE_CMAP, norm, SPLIT_LABELS[s], ylabels=(k == 0))
colorbar(fig, im, list(axes), "MAE (kcal/mol)", shrink=0.5, aspect=40)
save(fig, "heatmap_regression_MAE")
plt.show()

## 2. Классификация: ROC-AUC при ee ≥ 90%

In [ ]:
half_range = float(np.nanmax(np.abs(clf["ROC_AUC"] - 0.5)))
norm = TwoSlopeNorm(vcenter=0.5, vmin=0.5 - half_range, vmax=0.5 + half_range)

fig, axes = plt.subplots(1, 3, figsize=(FIG_WIDTH_MM * MM, FIG_HEIGHT_MM * MM), constrained_layout=True)
for k, (ax, s) in enumerate(zip(axes, SPLITS)):
    im = heat(ax, clf[clf.split == s], "ROC_AUC", CLF_MODELS, "RdBu", norm,
              f"{SPLIT_LABELS[s]} — ee ≥ {CLF_THRESHOLD}%", ylabels=(k == 0))
colorbar(fig, im, list(axes), "ROC-AUC", shrink=0.5, aspect=40)
save(fig, f"heatmap_classification_AUC_ee{CLF_THRESHOLD}")
plt.show()